In [ ]:
# ============================================================
# Integration: DeepSVDD (CNN) as Pre-Filter before MARL Agents
# ============================================================

from pathlib import Path
import joblib

# Load the already trained DeepSVDD + CNN (from notebook 03)
# Option A: If you saved it
# deep_svdd_cnn = joblib.load("results/deep_svdd_cnn.pkl")

# Option B: Recreate and load weights (recommended)
cnn_encoder = CNNEncoder1D(n_features=5, seq_len=30, embed_dim=32).to(device)
deep_svdd_cnn = DeepSVDD_CNN(cnn_encoder)
# deep_svdd_cnn.cnn.load_state_dict(torch.load("results/cnn_svdd_weights.pt"))
# deep_svdd_cnn.c = torch.load("results/svdd_center.pt")

def process_application_with_validation(
    tabular_features,          # [1, tabular_dim] or [batch, tabular_dim]
    sequence_features,         # [1, seq_len, n_features]
    applicant_id="APP_000",
    anomaly_threshold=None
):
    """
    Full pipeline:
    1. DeepSVDD anomaly check (CNN latent space)
    2. If normal → send to specialized MARL agents
    3. Return decision + explanation
    """

    # ---------- Stage 1: Anomaly Detection ----------
    preds, scores, thresh = deep_svdd_cnn.predict(
        sequence_features, threshold=anomaly_threshold
    )

    if preds[0] == 1:   # Anomalous
        explanation = {
            "applicant_id": applicant_id,
            "stage": "DATA_VALIDATION",
            "status": "FLAGGED_FOR_REVIEW",
            "anomaly_score": float(scores[0]),
            "threshold": float(thresh),
            "reason": "Sequence pattern significantly deviates from normal customers"
        }
        return "FLAGGED_FOR_REVIEW", explanation

    # ---------- Stage 2: MARL Specialized Agents ----------
    tabular_features = tabular_features.to(device).float()
    if tabular_features.dim() == 1:
        tabular_features = tabular_features.unsqueeze(0)

    agent_qs = []
    for name in AGENT_NAMES:
        slice_idx = feature_slices[name]
        q = agents[name](tabular_features[:, slice_idx])
        agent_qs.append(q)
    agent_qs = torch.stack(agent_qs)  # [num_agents, batch, action_dim]

    final_q, attn_weights = coordinator(agent_qs)
    action = final_q.argmax(dim=-1).item()

    # ---------- Stage 3: Explanation ----------
    contribution = attn_weights[0].mean(dim=0).detach().cpu().numpy()
    contribution_dict = {
        AGENT_NAMES[i]: round(float(contribution[i]), 4)
        for i in range(len(AGENT_NAMES))
    }
    top_agent = max(contribution_dict, key=contribution_dict.get)

    action_map = {0: "REJECTED", 1: "APPROVED", 2: "COUNTEROFFER"}
    explanation = {
        "applicant_id": applicant_id,
        "stage": "MARL_DECISION",
        "decision": action_map[action],
        "anomaly_score": float(scores[0]),
        "agent_contributions": contribution_dict,
        "primary_factor": top_agent,
        "explanation": (
            f"Application {applicant_id} was {action_map[action]}. "
            f"Primary driver: {top_agent} ({contribution_dict[top_agent]*100:.1f}%)."
        )
    }

    return action_map[action], explanation